# 8.2 Lab: Structured Output and Guided Decoding

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/harshuljain13/llm-inference-at-scale/blob/master/content/09_operations/08.2_structured_output/lab.ipynb) [![Open In Molab](https://img.shields.io/badge/Open%20in-Molab-blue)](https://molab.marimo.io/github/harshuljain13/llm-inference-at-scale/blob/master/content/09_operations/08.2_structured_output/lab.ipynb)

Benchmark constrained vs unconstrained decoding: measure conformance rates,
latency overhead of token masking, and validate structured output pipelines.

In [ ]:
# --- Setup: install dependencies for structured output experiments ---
import subprocess, sys
# Install pydantic for schema validation, numpy/matplotlib for benchmarks
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                       'numpy', 'matplotlib', 'pydantic'])

# JSON parsing and serialization
import json
# High-precision timing for latency measurement
import time
# Regular expression support for regex constraint validation
import re
# Numerical computing for statistics and simulation
import numpy as np
# Visualization for benchmark charts
import matplotlib.pyplot as plt
# Type annotations for function clarity
from typing import List, Optional, Literal
# Schema validation library (industry standard for structured LLM output)
from pydantic import BaseModel, Field, field_validator

In [ ]:
# --- Define Pydantic schemas that guided decoding enforces ---
# These models represent the JSON structure the LLM must produce

class PersonExtraction(BaseModel):
    """Schema for extracting person information from text."""
    # Person's full name (required, 1-100 chars)
    name: str = Field(..., min_length=1, max_length=100)
    # Age as integer (validated range prevents nonsense values)
    age: int = Field(..., ge=0, le=150)
    # Email with regex pattern validation
    email: str = Field(..., pattern=r'^[\w.-]+@[\w.-]+\.\w+$')
    # Skills list (must have at least one entry)
    skills: List[str] = Field(..., min_length=1)
    # Optional experience field
    experience_years: Optional[float] = Field(None, ge=0)


class FunctionCall(BaseModel):
    """Schema for LLM function/tool calling output."""
    # Function name must match snake_case pattern
    function_name: str = Field(..., pattern=r'^[a-z_][a-z0-9_]*$')
    # Arguments as key-value dictionary
    arguments: dict
    # Reasoning must be at least 10 chars (prevents empty justification)
    reasoning: str = Field(..., min_length=10)

    @field_validator('function_name')
    @classmethod
    def validate_function_exists(cls, v):
        """Ensure function name is in the allowed set."""
        # Only these 5 functions are valid tool calls
        allowed = {'search_web', 'get_weather', 'send_email', 'create_ticket', 'query_database'}
        if v not in allowed:
            raise ValueError(f'{v} not in allowed functions: {allowed}')
        return v


# Demonstrate successful validation
valid_person = PersonExtraction(
    name="Alice", age=30, email="alice@example.com", skills=["python", "ml"]
)
# Serialize to JSON to show the expected output format
print(f"Valid schema output:\n{valid_person.model_dump_json(indent=2)}")

In [ ]:
# --- Regex constraint validation: test format-specific patterns ---
# Guided decoding engines use regex FSMs to mask tokens at each step

# Define common regex patterns used in production
REGEX_PATTERNS = {
    # ISO 8601 date format (YYYY-MM-DD)
    "iso_date": r"\d{4}-(?:0[1-9]|1[0-2])-(?:0[1-9]|[12]\d|3[01])",
    # US phone number format ((XXX) XXX-XXXX)
    "phone_us": r"\(\d{3}\) \d{3}-\d{4}",
    # Semantic version (X.Y.Z with optional pre-release)
    "semver": r"\d+\.\d+\.\d+(?:-[a-zA-Z0-9.]+)?",
    # IPv4 address
    "ipv4": r"(?:\d{1,3}\.){3}\d{1,3}",
    # Hex color code
    "hex_color": r"#[0-9a-fA-F]{6}",
}

def validate_against_regex(output: str, pattern_name: str) -> dict:
    """Check if output matches the named regex pattern exactly."""
    # Retrieve the pattern for this constraint type
    pattern = REGEX_PATTERNS[pattern_name]
    # fullmatch requires the ENTIRE string to match (not just a substring)
    match = re.fullmatch(pattern, output.strip())
    # Return structured result with pass/fail
    return {"pattern": pattern_name, "output": output.strip(), "valid": match is not None}

# Test various outputs against their expected patterns
# Shows what constrained decoding prevents (invalid formats)
test_outputs = {
    "iso_date": ["2024-03-15", "2024-13-45", "March 15, 2024"],
    "phone_us": ["(555) 123-4567", "555-123-4567", "5551234567"],
    "semver": ["2.1.0", "2.1.0-beta.1", "v2.1"],
}

# Validate each test case and display results
for pattern_name, outputs in test_outputs.items():
    print(f"\n--- {pattern_name} ---")
    for out in outputs:
        result = validate_against_regex(out, pattern_name)
        # Show checkmark for valid, X for invalid
        status = '\u2713' if result['valid'] else '\u2717'
        print(f"  {status} '{out}'")

In [ ]:
# --- Constrained decoding simulation: measure token masking overhead ---
# Real engines (outlines, xgrammar) build FSMs from schemas to mask tokens
# This simulation measures the computational cost of the masking operation

class ConstrainedDecodingSimulator:
    """Simulates constrained vs unconstrained token sampling overhead."""

    def __init__(self, vocab_size: int = 32000):
        # Standard LLM vocabulary size (Llama-3 uses 128K but 32K is typical)
        self.vocab_size = vocab_size

    def unconstrained_step(self) -> int:
        """Sample one token without any constraints (baseline)."""
        # Generate random logits for full vocabulary
        logits = np.random.randn(self.vocab_size)
        # Softmax to get probability distribution
        probs = np.exp(logits) / np.exp(logits).sum()
        # Sample one token from the distribution
        return int(np.random.choice(self.vocab_size, p=probs))

    def constrained_step(self, valid_token_ratio: float = 0.1) -> int:
        """Sample with token masking (simulates FSM-guided decoding)."""
        # Generate logits for full vocabulary (same as unconstrained)
        logits = np.random.randn(self.vocab_size)
        # Build validity mask: True for tokens allowed by current FSM state
        mask = np.zeros(self.vocab_size, dtype=bool)
        # Number of valid tokens determined by constraint strictness
        n_valid = max(1, int(self.vocab_size * valid_token_ratio))
        # Randomly select which tokens are valid (simulates FSM lookup)
        valid_indices = np.random.choice(self.vocab_size, n_valid, replace=False)
        mask[valid_indices] = True
        # Set invalid token logits to -infinity (impossible to sample)
        logits[~mask] = -np.inf
        # Renormalize over only the valid tokens
        valid_logits = logits[mask]
        probs = np.exp(valid_logits) / np.exp(valid_logits).sum()
        # Sample from valid tokens only
        return int(valid_indices[np.random.choice(len(valid_indices), p=probs)])

    def benchmark(self, n_tokens: int = 200, valid_ratio: float = 0.1) -> dict:
        """Compare constrained vs unconstrained decoding latency."""
        # Time unconstrained decoding (baseline)
        t0 = time.perf_counter()
        for _ in range(n_tokens):
            self.unconstrained_step()
        unconstrained_ms = (time.perf_counter() - t0) * 1000

        # Time constrained decoding (with token masking)
        t0 = time.perf_counter()
        for _ in range(n_tokens):
            self.constrained_step(valid_ratio)
        constrained_ms = (time.perf_counter() - t0) * 1000

        # Calculate percentage overhead from constraints
        overhead = (constrained_ms / unconstrained_ms - 1) * 100
        return {
            "n_tokens": n_tokens,
            "valid_ratio": valid_ratio,
            "unconstrained_ms": round(unconstrained_ms, 2),
            "constrained_ms": round(constrained_ms, 2),
            "overhead_pct": round(overhead, 1),
        }

# Run a single benchmark comparison
# Initialize simulator with standard 32K vocabulary
sim = ConstrainedDecodingSimulator()
# Benchmark 200 tokens at 10% valid ratio (typical JSON schema strictness)
result = sim.benchmark(n_tokens=200, valid_ratio=0.1)
print("Single benchmark result:")
print(json.dumps(result, indent=2))

In [ ]:
# --- Sweep: measure overhead across different constraint strictness levels ---
# Lower valid_ratio = stricter constraint = fewer valid tokens per step

# Range from very strict (1% valid) to no constraint (100% valid)
ratios = [0.01, 0.05, 0.1, 0.2, 0.5, 0.8, 1.0]
# Collect benchmark results at each strictness level
# Accumulate results for each strictness level
sweep_results = []
for ratio in ratios:
    # Run 500 token decode at each ratio for stable measurement
    r = sim.benchmark(n_tokens=500, valid_ratio=ratio)
    sweep_results.append(r)
    # Print overhead at each level
    print(f"valid_ratio={ratio:.2f} -> overhead={r['overhead_pct']:.1f}%")

# Plot overhead vs constraint strictness
# Create figure for the overhead curve
fig_c4, ax_c4 = plt.subplots(1, 1, figsize=(8, 5))
# X-axis: valid token ratio (1.0 = no constraint, 0.01 = very strict)
ax_c4.plot(ratios, [r['overhead_pct'] for r in sweep_results], 'b-o', linewidth=2)
ax_c4.set_xlabel('Valid Token Ratio (1.0 = unconstrained, 0.01 = very strict)')
ax_c4.set_ylabel('Latency Overhead (%)')
ax_c4.set_title('Constrained Decoding Overhead vs Constraint Strictness')
# Zero line shows baseline (no overhead)
ax_c4.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
ax_c4.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# --- Conformance experiment: compare enforcement strategies ---
# Simulates what percentage of outputs are valid under each approach

def simulate_conformance(n_samples: int = 1000) -> dict:
    """Model conformance rates for different output enforcement strategies."""
    np.random.seed(42)

    # Empirical conformance rates from literature and production measurements
    strategies = {
        # No enforcement: model generates freely, ~62% valid JSON
        "unconstrained": {"rate": 0.62, "latency_mult": 1.0},
        # Schema in prompt: slightly better but unreliable
        "prompt_engineering": {"rate": 0.78, "latency_mult": 1.1},
        # Retry up to 3x on validation failure
        "retry_on_fail": {"rate": 0.91, "latency_mult": 1.8},
        # FSM-based token masking (regex/JSON constraints)
        "guided_decoding": {"rate": 0.99, "latency_mult": 1.15},
        # Full grammar enforcement (outlines CFG mode)
        "grammar_enforced": {"rate": 1.00, "latency_mult": 1.25},
    }

    # Simulate n_samples requests per strategy with binomial sampling
    # Store per-strategy results
results = {}
    for name, config in strategies.items():
        # Each sample is 1 (valid) or 0 (invalid) at the given rate
        samples = np.random.binomial(1, config["rate"], n_samples)
        results[name] = {
            # Observed conformance rate
            "conformance": samples.mean(),
            # Count of failed requests out of n_samples
            "failures": int((1 - samples).sum()),
            # Relative latency cost vs unconstrained baseline
            "latency_mult": config["latency_mult"],
        }
    return results

# Run the experiment and display results table
# Run with 1000 samples per strategy for statistical stability
conformance = simulate_conformance(1000)
print(f"{'Strategy':<22} {'Conformance':>11} {'Failures':>9} {'Latency':>8}")
print("-" * 55)
for name, data in conformance.items():
    # Format as table row with alignment
    print(f"{name:<22} {data['conformance']:>10.1%} {data['failures']:>9} {data['latency_mult']:>7.2f}x")

In [ ]:
# --- Visualization: conformance vs latency tradeoff scatter plot ---
fig_c6, (ax1_c6, ax2_c6) = plt.subplots(1, 2, figsize=(13, 5))

# Extract data for plotting
# Extract strategy names and metrics for plotting
names = list(conformance.keys())
rates = [conformance[s]['conformance'] * 100 for s in names]
latencies = [conformance[s]['latency_mult'] for s in names]
# Color palette matching the Excalidraw skill pastel colors
colors = ['#ef4444', '#f59e0b', '#3b82f6', '#10b981', '#8b5cf6']

# Left panel: bar chart of conformance rates per strategy
bars = ax1_c6.bar(range(len(names)), rates, color=colors, edgecolor='black', lw=0.5)
# Format x-axis labels with line breaks for readability
ax1_c6.set_xticks(range(len(names)))
ax1_c6.set_xticklabels([n.replace('_', '\n') for n in names], fontsize=9)
ax1_c6.set_ylabel('Conformance Rate (%)')
ax1_c6.set_title('Schema Conformance by Strategy')
# Set y-axis range to show differences clearly
ax1_c6.set_ylim(50, 105)
# Green dashed line at 100% (perfect conformance target)
ax1_c6.axhline(y=100, color='green', linestyle='--', alpha=0.5)
# Add value labels on top of each bar
for bar, rate in zip(bars, rates):
    ax1_c6.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
             f'{rate:.1f}%', ha='center', fontsize=9)

# Right panel: scatter plot showing conformance/latency tradeoff
ax2_c6.scatter(latencies, rates, c=colors, s=150, edgecolors='black', lw=1, zorder=5)
# Label each point with strategy name
for i, name in enumerate(names):
    ax2_c6.annotate(name.replace('_', ' '), (latencies[i], rates[i]),
                 textcoords="offset points", xytext=(10, 5), fontsize=8)
ax2_c6.set_xlabel('Latency Multiplier (vs unconstrained)')
ax2_c6.set_ylabel('Conformance Rate (%)')
ax2_c6.set_title('Conformance vs Latency Tradeoff')
ax2_c6.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# --- Pipeline benchmark: measure Pydantic validation overhead ---
# In production, Pydantic validation runs AFTER constrained decoding
# as defense-in-depth (belt and suspenders approach)

def pipeline_benchmark(n_requests: int = 200) -> dict:
    """Measure JSON parse + Pydantic validation latency per request."""
    # Track individual validation times in microseconds
    json_times = []
    pydantic_times = []

    for i in range(n_requests):
        # Simulate LLM output (already valid JSON from guided decoding)
        raw_output = json.dumps({
            "name": f"Person_{i}",
            "age": np.random.randint(18, 65),
            "email": f"person{i}@company.com",
            "skills": ["python", "ml"][:np.random.randint(1, 3)],
            "experience_years": round(np.random.uniform(0, 20), 1),
        })

        # Measure JSON parse time (deserialize string to dict)
        t0 = time.perf_counter()
        data = json.loads(raw_output)
        # Record in microseconds for precision
        json_times.append((time.perf_counter() - t0) * 1e6)

        # Measure Pydantic validation time (type checking + constraints)
        t0 = time.perf_counter()
        PersonExtraction(**data)
        # Record in microseconds
        pydantic_times.append((time.perf_counter() - t0) * 1e6)

    # Return P50 and P99 latencies for both stages
    return {
        "n_requests": n_requests,
        # JSON parse is typically < 10 microseconds
        "json_parse_p50_us": round(np.percentile(json_times, 50), 1),
        "json_parse_p99_us": round(np.percentile(json_times, 99), 1),
        # Pydantic validation is typically < 100 microseconds
        "pydantic_p50_us": round(np.percentile(pydantic_times, 50), 1),
        "pydantic_p99_us": round(np.percentile(pydantic_times, 99), 1),
    }

# Run the pipeline benchmark
pipe_results = pipeline_benchmark(200)
print("Validation Latency (microseconds):")
# Both stages add negligible overhead vs LLM generation (milliseconds)
print(json.dumps(pipe_results, indent=2))

In [ ]:
# --- Function calling validation: test tool-use schema enforcement ---
# In production, guided decoding constrains function_name to valid choices

# Define available tools (function calling registry)
TOOLS = {
    # Each tool has required parameters
    "search_web": ["query", "max_results"],
    "get_weather": ["city", "units"],
    "send_email": ["to", "subject", "body"],
}

def validate_function_call(raw_json: str) -> dict:
    """Validate a function call against Pydantic schema + tool registry."""
    try:
        # Parse raw JSON string into dictionary
        data = json.loads(raw_json)
        # Validate against FunctionCall schema (checks name pattern, reasoning)
        call = FunctionCall(**data)
        # Check that arguments contain all required parameters for the function
        required_args = TOOLS.get(call.function_name, [])
        # Find any missing required arguments
        missing = set(required_args) - set(call.arguments.keys())
        # Valid only if no required args are missing
        return {"valid": len(missing) == 0, "function": call.function_name, "missing": list(missing)}
    except Exception as e:
        # Any parse or validation error = invalid output
        return {"valid": False, "error": str(e)}

# Test cases: valid call, invalid function name, missing arguments
test_cases = [
    # Valid: correct function name and all required args present
    '{"function_name": "search_web", "arguments": {"query": "LLM inference", "max_results": 5}, "reasoning": "User wants to find information about LLM inference"}',
    # Invalid: function name not in allowed set
    '{"function_name": "hack_system", "arguments": {}, "reasoning": "Testing invalid function name"}',
    # Invalid: missing required 'subject' and 'body' arguments
    '{"function_name": "send_email", "arguments": {"to": "bob@x.com"}, "reasoning": "Missing required arguments for this function"}',
]

# Validate each test case and show results
print("Function call validation results:")
for call_json in test_cases:
    result = validate_function_call(call_json)
    # Checkmark for valid, X for invalid
    status = '\u2713' if result['valid'] else '\u2717'
    print(f"  {status} {result}")